To start the demo:
 - set environment variables
 - Open a terminal and run cyborgdb-service
 - You may have to wait for the servers to start running
 - Run All
 - At the bottom, open the url that ends with gradio.live

In [1]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "false"
%pip install --quiet --upgrade -q langchain==0.3.26 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.3.1 \
    langchain-openai==0.3.28 \
    cryptography==39.0.1 \
    pypdf==5.8.0 \
    openai==1.97.1 \
    gradio==5.38.0 \
    einops==0.8.1 \
    sentence-transformers==4.1.0 \
    ipywidgets \
    cyborgdb-service \
    cyborgdb
    # cyborgdb==0.11.0

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import langchain.chains
import langchain.chains.combine_documents
import langchain.docstore.document
import langchain.document_loaders
import langchain.prompts
import langchain.text_splitter
import langchain_community.chat_message_histories
import langchain_core.runnables.history
import langchain_huggingface
import langchain_openai
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64
import cyborgdb as cyborgdb
from cyborgdb.integrations.langchain import CyborgVectorStore
import os
import uuid
import gradio as gr
import getpass



In [3]:
EMBEDDING_MODEL_ID = "all-MiniLM-L6-v2"
QUERY_ENCODE_KWARGS = None
CYBORGDB_KEYS: dict[str, str] = {}
VECTORSTORE_STORE: dict[str, CyborgVectorStore] = {}
# Number of *characters*, not *tokens*, in a chunk
CHUNK_SIZE = 2048
# Number of *characters*, not *tokens*, to overlap between chunks
CHUNK_OVERLAP = 128
# We need a place to store the chat message histories and the chains for each user session.
# Keys are session IDs, values are the chat message histories and chains for that session.
CHAT_HISTORY_STORE: dict[
    str, langchain_community.chat_message_histories.ChatMessageHistory
] = {}
CHAINS_STORE: dict[
    str, langchain_core.runnables.history.RunnableWithMessageHistory
] = {}
CYBORGDB_API_KEY = os.environ.get("CYBORGDB_API_KEY") or getpass.getpass("Enter CYBORGDB_API_KEY: ")
CYBORGDB_HOST = "http://localhost:8000"
OPENAI_URL = "http://localhost:11434/v1"  # Or wherever your LLM server is
OPENAI_MODEL = "phi3:mini"

In [ ]:
def split_documents(
    file_path: str,
) -> list[langchain.docstore.document.Document]:
    """Split a document into smaller pieces for processing.

    LangChain has many different types of document loaders. For brevity, we will
    only use the CSVLoader, the PyPDFLoader, and the TextLoader.

    Args:
        file_path: Path to the uploaded file (specified by gradio).

    Returns:
        A list of documents, each containing a chunk of the original document.

    Raises:
        ValueError: If the file is not specified.
    """
    file = pathlib.Path(file_path)
    if not file or not file.exists():
        raise ValueError("File is required")

    # Switch over file types to determine how to load the document
    if file.suffix == ".csv":
        loader = langchain.document_loaders.CSVLoader(file)
        # There is no need to split the document if it is a CSV file, each
        # row will be treated as a separate document automatically.
        documents = loader.load()
    elif file.suffix == ".pdf":
        loader = langchain.document_loaders.PyPDFLoader(str(file))
        pages = loader.load()
        
        text_splitter = langchain.text_splitter.RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        documents = text_splitter.split_documents(pages)
    else:
        loader = langchain.document_loaders.TextLoader(file)
        pages = loader.load()
        text_splitter = langchain.text_splitter.RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        documents = text_splitter.split_documents(pages)

    return documents

In [5]:
def encrypt_chunk(text: str, key: bytes) -> str:
    """Encrypt a text chunk using AES-256-GCM."""
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)  # 12 bytes for GCM
    ciphertext = aesgcm.encrypt(nonce, text.encode(), None)
    # Combine nonce + ciphertext and encode as base64 for display
    encrypted_data = nonce + ciphertext
    return base64.b64encode(encrypted_data).decode()

def upload_and_create_cyborg_vector_store(
    file_paths: list[str],
    embedding_model_id: str,
    session_id: str,
    preview_chars: int = 64,
    max_previews: int = 10,
) -> str:
    """Upload files to CyborgDB vector store with encryption preview."""
    try:
        if not file_paths:
            return "No files selected"
        
        print(f"DEBUG: Processing {len(file_paths)} files for session {session_id}")
                    
        # Get or create the vector store for this session
        vectorstore = load_cyborgdb_vectorstore(session_id, embedding_model_id)
        
        # Get the encryption key for this session
        encryption_key = bytes(CYBORGDB_KEYS[session_id])
        
        # Process and add documents
        all_documents = []
        encrypted_previews = []
        
        for file_path in file_paths:
            print(f"DEBUG: Processing file: {file_path}")
            documents = split_documents(file_path)
            print(f"DEBUG: Split into {len(documents)} chunks")
            
            # Encrypt each chunk and create previews
            for i, doc in enumerate(documents):
                # Encrypt the chunk content
                encrypted_content = encrypt_chunk(doc.page_content, encryption_key)
                
                # Store preview of encrypted content (first N characters)
                if len(encrypted_previews) < max_previews:
                    preview = encrypted_content[:preview_chars]
                    encrypted_previews.append({
                        'file': os.path.basename(file_path),
                        'chunk_index': i,
                        'original_length': len(doc.page_content),
                        'encrypted_length': len(encrypted_content),
                        'encrypted_preview': preview
                    })
                
                print(f"DEBUG: Chunk {i}: Original={len(doc.page_content)} chars, Encrypted={len(encrypted_content)} chars")
            
            all_documents.extend(documents)
        
        if all_documents:
            print(f"DEBUG: Adding {len(all_documents)} documents to vector store")
            vectorstore.add_documents(all_documents)
            
            # Train the index
            print("DEBUG: Training index...")
            vectorstore.index.train()
            print("DEBUG: Index trained successfully")
            
            # Test retrieval immediately after adding
            print("DEBUG: Testing retrieval...")
            test_results = vectorstore.similarity_search("test", k=3)
            print(f"DEBUG: Retrieved {len(test_results)} documents for test query")
        
        # Create the response with encryption previews
        response_lines = [
            f"Successfully processed {len(file_paths)} files with {len(all_documents)} chunks",
            "",
        ]
        
        for i, preview in enumerate(encrypted_previews):
            response_lines.extend([
                f"Chunk {i+1}: {preview['encrypted_preview']}...",
                ""
            ])
        
        if len(all_documents) > max_previews:
            response_lines.append(f"... and {len(all_documents) - max_previews} more encrypted chunks")
        
        return "\n".join(response_lines)
        
    except Exception as e:
        print(f"DEBUG: Error in upload_and_create_cyborg_vector_store: {str(e)}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}"
    
def load_chain(
    session_id: str,
    system_prompt: str,
    embedding_model_id: str,
    openai_url: str | None,
    openai_model: str,
) -> langchain_core.runnables.history.RunnableWithMessageHistory:
    """Create a chain for retrieving answers to questions from CyborgDB and prompting the LLM."""

    print(f"DEBUG: Loading chain for session {session_id}")
    call_llm = langchain_openai.ChatOpenAI(
        base_url=openai_url, model=openai_model, max_tokens=500, temperature=0.7
    )
    
    # Get the CyborgDB vector store for this session
    vectordb = load_cyborgdb_vectorstore(session_id, embedding_model_id)
    
    # Test retrieval before creating retriever
    print("DEBUG: Testing vector store retrieval in chain...")
    test_docs = vectordb.similarity_search("test", k=1)
    print(f"DEBUG: Vector store has {len(test_docs)} documents for test query")
    
    retriever = vectordb.as_retriever(search_kwargs={"k": 3})

    # Ask the LLM to reformulate user prompts as queries for the document store
    contextualize_question_system_prompt = (
        "Given a chat history and the latest user question "
        "which might reference context in the chat history, formulate a standalone question "
        "which can be understood without the chat history. If the question is not related to the chat history, "
        "leave the question intact. Do NOT answer the question, "
        "just reformulate it if needed and otherwise return it as is."
    )
    contextualize_question_prompt = (
        langchain.prompts.ChatPromptTemplate.from_messages(
            [
                ("system", contextualize_question_system_prompt),
                langchain.prompts.MessagesPlaceholder("chat_history"),
                ("human", "{input}"),
            ]
        )
    )

    # Retrieve the most relevant documents for a given question from the vector database
    history_aware_retriever = langchain.chains.create_history_aware_retriever(
        call_llm, retriever, contextualize_question_prompt
    )

    # Create a new prompt including the chat history and the retrieved documents
    document_prompt = langchain.prompts.PromptTemplate(
        input_variables=["page_content", "source"],
        template="Context:\n{page_content}\n\nSource: {source}",
    )
    user_chat_prompt = langchain.prompts.ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            langchain.prompts.MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
        ]
    )

    # Query the LLM with the chat history and the most relevant documents
    user_chat_chain_with_documents = (
        langchain.chains.combine_documents.create_stuff_documents_chain(
            call_llm, user_chat_prompt, document_prompt=document_prompt
        )
    )

    # Compose the retrieval and chat chains
    rag_chain = langchain.chains.create_retrieval_chain(
        history_aware_retriever, user_chat_chain_with_documents
    )

    # Create a chain that stores the chat message history for the session, using the RAG chain
    conversation_rag_chain = (
        langchain_core.runnables.history.RunnableWithMessageHistory(
            rag_chain,
            get_session_history=lambda: CHAT_HISTORY_STORE.get(
                session_id,
                langchain_community.chat_message_histories.ChatMessageHistory(),
            ),
            input_messages_key="input",
            history_messages_key="chat_history",
            output_messages_key="answer",
        )
    )

    print("DEBUG: Chain loaded successfully")
    return conversation_rag_chain

def load_cyborgdb_vectorstore(
    session_id: str,
    embedding_model_id: str,
) -> CyborgVectorStore:
    """Load a CyborgDB vector store with caching."""
    try:
        # Return cached instance if exists
        if session_id in VECTORSTORE_STORE:
            print(f"DEBUG: Returning cached vector store for session {session_id}")
            return VECTORSTORE_STORE[session_id]

        # Generate a unique index key for this session
        if session_id not in CYBORGDB_KEYS:
            index_key = CyborgVectorStore.generate_key()
            CYBORGDB_KEYS[session_id] = index_key
            print(f"DEBUG: Generated new index key for session {session_id}")
        else:
            index_key = CYBORGDB_KEYS[session_id]
            print(f"DEBUG: Using existing index key for session {session_id}")

        # Create the vector store - it will automatically create the index if needed
        vector_store = CyborgVectorStore(
            index_name=f"session_{session_id}",
            index_key=index_key,
            api_key=CYBORGDB_API_KEY,
            base_url=CYBORGDB_HOST,
            embedding=embedding_model_id,
            index_type="ivfflat",
            metric="cosine",
        )

        print(f"DEBUG: Created CyborgVectorStore for session {session_id}")

        # Cache it before returning
        VECTORSTORE_STORE[session_id] = vector_store
        return vector_store
    except Exception as e:
        print(f"DEBUG: Error creating vector store: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

In [6]:

MOST_RECENT_PROMPT: str | None = None


def fetch_session_hash(request: gr.Request) -> str | None:
    """Fetch the session hash from the request.

    We will use the session hash as the CyborgDB collection name so that each
    individual session has its own collection in CyborgDB.

    Args:
        request: The Gradio request object.

    Returns:
        The session hash.
    """
    return request.session_hash


def chat_bot(
    message: str,
    history: list[list[str]],
    session: str,
    system_prompt: str,
    openai_url: str | None,
    openai_model: str,
    embedding_model_id: str,
) -> str:
    """Chat with the RAG conversation agent.

    Args:
        message: The message from the user.
        history: The chat history. Unused, because langchain handles its own copy of the chat history.
        session: The session hash.
        system_prompt: The system prompt.
        openai_url: The URL of the OpenAI API.
        openai_model: The OpenAI model to use for answering questions.
        embedding_model_id: The HuggingFace model ID to use for embedding the documents.

    Returns:
        The response from the RAG conversation agent.
    """
    global MOST_RECENT_PROMPT
    MOST_RECENT_PROMPT = message

    if session not in CHAINS_STORE:
        # Initialize chat history for this session
        if session not in CHAT_HISTORY_STORE:
            CHAT_HISTORY_STORE[session] = langchain_community.chat_message_histories.ChatMessageHistory()
        
        CHAINS_STORE[session] = load_chain(
            session,
            system_prompt,
            embedding_model_id,
            openai_url,
            openai_model,
        )
    
    chain = CHAINS_STORE[session]
    
    print(f"\n===== Query: '{message}' =====")
    response = chain.invoke(
        {"input": message}, config={"configurable": {"session_id": session}}
    )
    
    # Log what context was retrieved
    if 'context' in response:
        print(f"DEBUG: Retrieved {len(response['context'])} context documents")
        for i, doc in enumerate(response['context']):
            content = doc.page_content if hasattr(doc, 'page_content') else str(doc)
            print(f"DEBUG: Context Doc {i+1} (first 300 chars): {content[:300]}...")
    else:
        print("DEBUG: WARNING - No context in response")
    
    print(f"DEBUG: Answer (first 500 chars): {response['answer'][:500]}...")
    print("===== End Query =====\n")
    
    return response["answer"]

def setup_cyborg_chatbot_web_app(
    openai_url: str | None,
    openai_model: str,
) -> gr.Blocks:
    """Set up the Gradio interface for the RAG conversation agent.

    Args:
        cyborg_host: The host of the CyborgDB server.
        openai_url: The URL of the OpenAI API.
        openai_model: The OpenAI model to use for answering questions.
        show_reconstructed_prompt: Whether to show the reconstructed prompt.

    Returns:
        The Gradio interface for the RAG conversation agent.
    """
    with gr.Blocks() as demo, gr.Row():
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("## RAG Conversation Agent with CyborgDB")
            system_prompt = gr.Textbox(
                label="System instruction",
                lines=3,
                value="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Keep the answer concise. {context}",
            )

            session = gr.Textbox(value=str(uuid.uuid4()), label="Session")
            chat_interface = gr.ChatInterface(
                fn=lambda message, history, session, system_prompt: chat_bot(
                    message,
                    history,
                    session,
                    system_prompt,
                    openai_url,
                    openai_model,
                    EMBEDDING_MODEL_ID,
                ),
                additional_inputs=[
                    session,
                    system_prompt,
                ],
            )
            demo.load(fetch_session_hash, None, session)

        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("## Upload Document")
            file = gr.File(type="filepath", file_count="multiple")
            with gr.Row(equal_height=True), gr.Column(variant="compact"):
                create_vector_store_button = gr.Button(
                    "Create vector store", variant="primary", scale=1
                )
                vector_index_msg_out = gr.Textbox(
                    show_label=False,
                    lines=1,
                    scale=1,
                    placeholder="Please create vector store...",
                )

            create_vector_store_button.click(
                lambda files, session_id: upload_and_create_cyborg_vector_store(
                    files if files else [],
                    EMBEDDING_MODEL_ID,
                    session_id,
                ),
                inputs=[file, session],
                outputs=[vector_index_msg_out],
            )
            gr.HTML(
                """
                <style>
                    #my_image img {
                        border: 1px solid #e4e4e7;  /* green border, change color as needed */
                        border-radius: 8px;         /* rounded corners */
                        padding: 4px;               /* space between border and image */
                        background-color: none;    /* optional background */
                    }
                </style>
                """
            )
            gr.Image(
                value='EncryptedRagChatbot.png',
                show_label=False,
                interactive=False,
                height=280,
                elem_id="my_image"
            )

        return demo

In [ ]:
protected_demo = setup_cyborg_chatbot_web_app(
    openai_url=OPENAI_URL,
    openai_model=OPENAI_MODEL,
)

protected_demo.launch(inline=False, share=True)

/Users/jamesarmbruster/miniconda3/envs/3.12env/lib/python3.12/site-packages/gradio/chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1222018f85a821675b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


SSL verification is disabled. Not recommended for production.


DEBUG: Processing 1 files for session mb2nxjr7497
DEBUG: Generated new index key for session mb2nxjr7497
DEBUG: Created CyborgVectorStore for session mb2nxjr7497
DEBUG: Processing file: /private/var/folders/rv/p_k42pyn1_lbvyqsktkzm_br0000gn/T/gradio/9a55d65e7385f427fa68c0b1405b097a57b481e1d02d3d5ad2c347b9d4985d8c/HMF_Lease_End_Kit.pdf
DEBUG: Split into 9 chunks
DEBUG: Chunk 0: Original=90 chars, Encrypted=160 chars
DEBUG: Chunk 1: Original=663 chars, Encrypted=932 chars
DEBUG: Chunk 2: Original=737 chars, Encrypted=1044 chars
DEBUG: Chunk 3: Original=864 chars, Encrypted=1208 chars
DEBUG: Chunk 4: Original=1429 chars, Encrypted=1956 chars
DEBUG: Chunk 5: Original=758 chars, Encrypted=1056 chars
DEBUG: Chunk 6: Original=1978 chars, Encrypted=2680 chars
DEBUG: Chunk 7: Original=1539 chars, Encrypted=2108 chars
DEBUG: Chunk 8: Original=13 chars, Encrypted=56 chars
DEBUG: Adding 9 documents to vector store
DEBUG: Training index...
DEBUG: Index trained successfully
DEBUG: Testing retrieval.

SSL verification is disabled. Not recommended for production.


DEBUG: Processing 3 files for session 1cpspvpemgv
DEBUG: Generated new index key for session 1cpspvpemgv
DEBUG: Created CyborgVectorStore for session 1cpspvpemgv
DEBUG: Processing file: /private/var/folders/rv/p_k42pyn1_lbvyqsktkzm_br0000gn/T/gradio/a868f9e94eb3fe82c171d7be2f4427465f6a88a870c228d60b8f4602d819fdc8/Janices mom.pdf
DEBUG: Split into 9 chunks
DEBUG: Chunk 0: Original=2044 chars, Encrypted=2776 chars
DEBUG: Chunk 1: Original=918 chars, Encrypted=1272 chars
DEBUG: Chunk 2: Original=2044 chars, Encrypted=2792 chars
DEBUG: Chunk 3: Original=2044 chars, Encrypted=2764 chars
DEBUG: Chunk 4: Original=515 chars, Encrypted=724 chars
DEBUG: Chunk 5: Original=2043 chars, Encrypted=2764 chars
DEBUG: Chunk 6: Original=2045 chars, Encrypted=2772 chars
DEBUG: Chunk 7: Original=568 chars, Encrypted=796 chars
DEBUG: Chunk 8: Original=628 chars, Encrypted=880 chars
DEBUG: Processing file: /private/var/folders/rv/p_k42pyn1_lbvyqsktkzm_br0000gn/T/gradio/58ed05db09be925f4945244a9fcf21ea34667bb

SSL verification is disabled. Not recommended for production.


DEBUG: Processing 3 files for session opbwvuz58ar
DEBUG: Generated new index key for session opbwvuz58ar
DEBUG: Created CyborgVectorStore for session opbwvuz58ar
DEBUG: Processing file: /private/var/folders/rv/p_k42pyn1_lbvyqsktkzm_br0000gn/T/gradio/442f3930fad0f0a1c398136f58f07061bd3ed740866956eeacd4275663e5005c/Janices mom.txt
DEBUG: Split into 6 chunks
DEBUG: Chunk 0: Original=46 chars, Encrypted=104 chars
DEBUG: Chunk 1: Original=1591 chars, Encrypted=2168 chars
DEBUG: Chunk 2: Original=1769 chars, Encrypted=2416 chars
DEBUG: Chunk 3: Original=1968 chars, Encrypted=2664 chars
DEBUG: Chunk 4: Original=1815 chars, Encrypted=2460 chars
DEBUG: Chunk 5: Original=1823 chars, Encrypted=2468 chars
DEBUG: Processing file: /private/var/folders/rv/p_k42pyn1_lbvyqsktkzm_br0000gn/T/gradio/de5f9142de72b9a3750606a697e5274c5c9f756283925e790a37de6a0a8c0f6d/Re_ Janice What is one of your earliest childhood memories.txt
DEBUG: Split into 4 chunks
DEBUG: Chunk 0: Original=1712 chars, Encrypted=2332 ch